# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

I rank content by CTR opportunity and search exposure.

A page receives a higher score when its observed CTR is below the expected CTR for its position and it has enough impressions to represent a meaningful opportunity.

The rule is intended as a simple decision-support baseline, not as a prediction of future performance.

Pages with stronger CTR opportunity and higher exposure are prioritized first.

In [8]:
!git clone https://github.com/06sushmita/flyrank-ML-Internship.git

Cloning into 'flyrank-ML-Internship'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 199 (delta 84), reused 177 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 1.89 MiB | 8.15 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [9]:
%cd /content/flyrank-ML-Internship

/content/flyrank-ML-Internship


In [10]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [11]:
!ls data/raw

content_refresh_anonymized.csv


In [12]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [13]:
import pandas as pd
import numpy as np

# Load the bundled anonymized dataset
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


### Signal 1 — CTR relative to position

CTR should be interpreted together with average position. A low CTR at a poor search position may be normal, while a low CTR at a strong position may represent a more useful opportunity.

This signal is linked to the CTR-fix logic discussed in the session.

I will check whether observed CTR changes directionally across position buckets before using CTR-vs-position in the baseline rule.

In [14]:
signal_df = df[
    ["content_id", "ctr", "avg_position", "impressions_90d"]
].copy()

signal_df = signal_df.dropna(
    subset=["ctr", "avg_position", "impressions_90d"]
)

signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

signal_1 = (
    signal_df
    .groupby("position_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

display(signal_1)

,position_bucket,n,median_ctr,mean_ctr,median_impressions
0,1-3,2346,0.00,1.472869,3.0
1,4-10,11842,0.16,0.651045,1184.0
2,11-20,7273,0.10,0.323443,870.0
3,21+,8539,0.00,0.211333,648.0


### Signal 2 — Search exposure

Impressions represent observed search exposure. If a page has a CTR opportunity and substantial exposure, prioritizing it may provide more useful decision support than prioritizing a low-exposure page.

This signal is related to the volume/quick-win logic discussed in the session.

I will check the distribution of pages and CTR across exposure buckets before using impressions in the baseline rule.

In [15]:
signal_df["impression_bucket"] = pd.qcut(
    signal_df["impressions_90d"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

signal_2 = (
    signal_df
    .groupby("impression_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(signal_2)

,impression_bucket,n,median_impressions,median_ctr,median_position
0,Low,10003,22.0,0.0,8.5
1,Medium,9997,731.0,0.1,14.8
2,High,10000,6641.0,0.2,9.3


### Signal verdicts

**CTR vs position — CONFIRMED**

Observed CTR decreases directionally as average position moves into weaker position buckets. This supports using position as context when evaluating CTR opportunity.

**Impressions — CONFIRMED**

Observed impression volume provides a useful exposure signal for prioritization. Higher-exposure pages represent a larger observed search opportunity when a CTR gap exists.

### Reason codes

The baseline will output one reason code per page:

- `LOW_CTR_HIGH_EXPOSURE` — observed CTR is below the expected CTR for its position and exposure is relatively high.
- `LOW_CTR_MED_EXPOSURE` — observed CTR is below the expected CTR for its position and exposure is moderate.
- `NO_CLEAR_OPPORTUNITY` — the observed signals do not indicate a strong enough opportunity for this baseline.

### Final baseline rule

For each page, estimate an expected CTR from pages in the same position bucket. Calculate the CTR gap as expected CTR minus observed CTR.

Combine the positive CTR gap with the page's relative impression exposure to produce an opportunity score.

Rank pages from highest to lowest score.

High-scoring pages receive an `OPTIMIZE_CTR` action, while moderate opportunities receive `REVIEW_CTR`. Pages without a clear opportunity receive `NO_ACTION`.

This is a transparent, directional decision-support baseline and is not a prediction of future performance.

# 2. **Build the ranked queue (writes the CSV)**
Code the score, rank everything, write work/outputs/baseline_action_score.csv.

In [16]:
score_df = df[
    [
        "content_id",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
].copy()

score_df = score_df.dropna(
    subset=[
        "content_id",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
)

score_df = score_df[
    (score_df["avg_position"] > 0) &
    (score_df["impressions_90d"] >= 0)
].copy()

print("Rows used for scoring:", len(score_df))

Rows used for scoring: 28795


In [18]:
score_df["position_bucket"] = pd.cut(
    score_df["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

In [19]:
expected_ctr = (
    score_df
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
    .rename("expected_ctr")
)

score_df = score_df.join(
    expected_ctr,
    on="position_bucket"
)

In [20]:
score_df["ctr_gap"] = (
    score_df["expected_ctr"] - score_df["ctr"]
)

score_df["positive_ctr_gap"] = (
    score_df["ctr_gap"].clip(lower=0)
)

In [21]:
score_df["exposure_score"] = (
    score_df["impressions_90d"]
    .rank(pct=True)
)

In [22]:
score_df["opportunity_score"] = (
    score_df["positive_ctr_gap"]
    * score_df["exposure_score"]
)

In [23]:
score_df["reason_code"] = np.select(
    [
        (
            (score_df["positive_ctr_gap"] > 0) &
            (score_df["exposure_score"] >= 0.67)
        ),
        (
            (score_df["positive_ctr_gap"] > 0) &
            (score_df["exposure_score"] >= 0.33)
        )
    ],
    [
        "LOW_CTR_HIGH_EXPOSURE",
        "LOW_CTR_MED_EXPOSURE"
    ],
    default="NO_CLEAR_OPPORTUNITY"
)

In [24]:
score_df["action"] = np.select(
    [
        score_df["reason_code"] == "LOW_CTR_HIGH_EXPOSURE",
        score_df["reason_code"] == "LOW_CTR_MED_EXPOSURE"
    ],
    [
        "OPTIMIZE_CTR",
        "REVIEW_CTR"
    ],
    default="NO_ACTION"
)

In [25]:
ranked = (
    score_df
    .sort_values(
        "opportunity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked["rank"] = ranked.index + 1

In [26]:
baseline_output = ranked[
    [
        "rank",
        "content_id",
        "opportunity_score",
        "reason_code",
        "action"
    ]
].copy()

display(baseline_output.head(20))

,rank,content_id,opportunity_score,reason_code,action
0,1,content_c8e9d6ab9013,0.159844,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
1,2,content_f986bd514b6e,0.151426,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
2,3,content_453722754fea,0.149599,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
3,4,content_39881853ef0c,0.149276,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
4,5,content_d274ac4158ef,0.148088,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
5,6,content_825a9788af8d,0.147873,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
6,7,content_e5f459e737b7,0.147708,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
7,8,content_8ba781dafa55,0.147376,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
8,9,content_339b357d04c7,0.147057,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR
9,10,content_5d5653c4eb4f,0.146592,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR


In [27]:
from pathlib import Path

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_output.to_csv(
    output_path,
    index=False
)

print(f"Saved {len(baseline_output):,} ranked rows to {output_path}")

Saved 28,795 ranked rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [28]:
top20 = ranked.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_id",
            "opportunity_score",
            "reason_code",
            "action",
            "ctr",
            "expected_ctr",
            "avg_position",
            "impressions_90d"
        ]
    ]
)

,rank,content_id,opportunity_score,reason_code,action,ctr,expected_ctr,avg_position,impressions_90d
0,1,content_c8e9d6ab9013,0.159844,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.00,0.16,9.7,208678
1,2,content_f986bd514b6e,0.151426,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.00,0.16,6.6,22456
2,3,content_453722754fea,0.149599,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.01,0.16,7.6,140079
3,4,content_39881853ef0c,0.149276,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.01,0.16,7.2,112434
4,5,content_d274ac4158ef,0.148088,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.01,0.16,6.8,65138
5,6,content_825a9788af8d,0.147873,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.00,0.16,5.6,16786
6,7,content_e5f459e737b7,0.147708,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.01,0.16,5.9,56363
7,8,content_8ba781dafa55,0.147376,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.00,0.16,9.0,16156
8,9,content_339b357d04c7,0.147057,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.01,0.16,3.7,46879
9,10,content_5d5653c4eb4f,0.146592,LOW_CTR_HIGH_EXPOSURE,OPTIMIZE_CTR,0.00,0.16,5.7,15101


### Top-20 review

1. **Action:** OPTIMIZE_CTR | **Reason:** LOW_CTR_HIGH_EXPOSURE | **Confidence:** High — large exposure and a noticeable CTR gap. | **Could be wrong if:** the page serves a navigational query where lower CTR is expected.

2. **Action:** OPTIMIZE_CTR | **Reason:** LOW_CTR_HIGH_EXPOSURE | **Confidence:** Medium — the score is supported by exposure and CTR gap. | **Could be wrong if:** the page was recently changed and the observed CTR is not stable.

...

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [29]:
display(
    ranked.tail(10)[
        [
            "rank",
            "content_id",
            "opportunity_score",
            "reason_code",
            "action",
            "ctr",
            "expected_ctr",
            "avg_position",
            "impressions_90d"
        ]
    ]
)

,rank,content_id,opportunity_score,reason_code,action,ctr,expected_ctr,avg_position,impressions_90d
28785,28786,content_7ccec5507540,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.00,0.00,39.5,983
28786,28787,content_74cd003b018c,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.24,0.10,13.8,7761
28787,28788,content_9300fbb3ff7b,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,2.03,0.16,9.2,17104
28788,28789,content_7299ec98cd1e,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.48,0.16,8.1,1042
28789,28790,content_567c6448b441,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.04,0.00,35.1,10884
28790,28791,content_663be1d5e31c,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,33.33,0.00,0.3,3
28791,28792,content_2859418b01d8,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.31,0.16,4.1,321
28792,28793,content_6c50f37e5d2f,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.77,0.16,5.3,10613
28793,28794,content_294ae0707486,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.00,0.00,23.1,37
28794,28795,content_a48b2c34f432,0.0,NO_CLEAR_OPPORTUNITY,NO_ACTION,0.13,0.10,18.2,1500


### Weak picks

The weakest picks show where the simple baseline can be unreliable. A high exposure value alone does not establish that changing a page will improve CTR. Similarly, a large CTR gap can be noisy when the observed exposure is limited.

These cases show why the baseline is a decision-support rule rather than a causal prediction.

### Leakage check

The score uses observed CTR, average position, and impressions from the selected dataset window. No future-period performance, future labels, or product flags are used as scoring inputs.

The baseline therefore uses signals available at the decision point.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.